In [2]:
import twixtools
import math
import numpy as np
import json
import os
home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
# cd to data folder
os.chdir(parent_folder)

data_folder = os.path.join(parent_folder, "data/datasets/msk_mri")


name = "meas_MID00063_FID126620_AX_PD_FS"
filename = os.path.join(data_folder, name + ".dat")
raw_file = twixtools.read_twix(filename)[-1]

Software version: VD/VE (!?)

Scan  0


100%|██████████| 29.0M/29.0M [00:00<00:00, 586MB/s]


Scan  1


100%|██████████| 595M/595M [00:00<00:00, 1.05GB/s]


In [ ]:

def is_clean_metadata(key, value):
    # Exclude keys that are likely to contain large data or geometry/weird stuff
    exclude_keys = ["mdb", "data", "geometry", "image", "buffer", "raw", "array", "twix", "kspace"]
    if any(ex in key.lower() for ex in exclude_keys):
        return False
    # Exclude numpy arrays, torch tensors, and other large objects
    if hasattr(value, 'shape') or hasattr(value, 'dtype'):
        return False
    # Exclude lists/tuples with more than 100 elements
    if isinstance(value, (list, tuple)) and len(value) > 100:
        return False
    return True


def extract_clean_metadata(obj):
    if isinstance(obj, dict):
        return {k: extract_clean_metadata(v) for k, v in obj.items() if is_clean_metadata(k, v)}
    elif hasattr(obj, '__dict__'):
        return extract_clean_metadata(vars(obj))
    elif isinstance(obj, (list, tuple)):
        if len(obj) > 100:
            return f"List/Tuple of length {len(obj)} omitted"
        return [extract_clean_metadata(v) for v in obj]
    else:
        # Preserve primitives; convert numpy scalars
        if isinstance(obj, (str, bool, int, float)) or obj is None:
            return obj
        if isinstance(obj, np.generic):
            return obj.item()
        try:
            return str(obj)
        except Exception:
            return None
        
def _is_zero_like(x):
    # numbers (but not bools)
    if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool):
        return x == 0
    # numpy arrays
    if isinstance(x, np.ndarray):
        return x.size > 0 and np.all(x == 0)
    # containers: all elements zero-like
    if isinstance(x, (list, tuple, set)):
        return len(x) > 0 and all(_is_zero_like(el) for el in x)
    return False

def _is_empty_value(x):
    # treat zero/zero-like as empty
    if _is_zero_like(x):
        return True
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    if isinstance(x, np.floating) and np.isnan(x):
        return True
    if isinstance(x, str):
        s = x.strip()
        return s == "" or s.lower() == "nan"
    # empty containers or containers whose items are all empty
    if isinstance(x, (list, tuple, set)):
        return len(x) == 0 or all(_is_empty_value(i) for i in x)
    if isinstance(x, dict):
        return len(x) == 0 or all(_is_empty_value(v) for v in x.values())
    return False

def prune_empty(obj):
    if isinstance(obj, dict):
        pruned = {}
        for k, v in obj.items():
            pv = prune_empty(v)
            if pv is not None and not _is_empty_value(pv):
                pruned[k] = pv
        return pruned if pruned else None
    if isinstance(obj, (list, tuple, set)):
        out = []
        for item in obj:
            pi = prune_empty(item)
            if pi is not None and not _is_empty_value(pi):
                out.append(pi)
        return out if out else None
    return None if _is_empty_value(obj) else obj

clean_metadata = {}
for key in raw_file.keys():
    value = raw_file[key]
    if is_clean_metadata(key, value):
        clean_metadata[key] = extract_clean_metadata(value)

# Prune empty strings, NaNs, None, and empty containers
clean_metadata_nonempty = prune_empty(clean_metadata) or {}
# save on home folder
save_path = os.path.join(home_folder, "msk_mri_dataset", 'raw_file_metadata_' + name + '.json')

with open(save_path, 'w') as f:
    json.dump(clean_metadata_nonempty, f, indent=2)

print('Saved clean raw_file metadata (non-empty only) to raw_file_metadata_' + name + '.json')

PermissionError: [Errno 13] Permission denied: 'raw_file_metadata_meas_MID00063_FID126620_AX_PD_FS.json'

In [5]:
# Save a JSON file with all keys and subkeys (nested structure) from non-empty metadata
def keys_structure(obj):
    if isinstance(obj, dict):
        return {k: keys_structure(v) for k, v in obj.items()}
    elif hasattr(obj, '__dict__'):
        return keys_structure(vars(obj))
    elif isinstance(obj, (list, tuple)):
        return [] if len(obj) == 0 else [keys_structure(obj[0])]
    else:
        return None  # just structure

all_keys_structure = keys_structure(clean_metadata_nonempty)
with open(save_path + 'raw_file_all_keys.json', 'w') as f:
    json.dump(all_keys_structure, f, indent=2)
print('Saved keys/subkeys structure (non-empty only) to raw_file_all_keys.json')

Saved keys/subkeys structure (non-empty only) to raw_file_all_keys.json
